In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
import torchvision.transforms as transforms




transform = transforms.Compose([
    transforms.Resize((28, 28)),
    # Task: نسوي Resize الي 28x28
    transforms.Grayscale(3),
    # تحويل grayscale to RGB
    transforms.ToTensor(),

    # Task: تحويل اليTensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Task: Normalize
])

# Load EMNIST dataset
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

num_classes = 26

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
# Write your code here

from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
import torch.nn as nn


# نستخدم EfficientNet_V2 عشان نضمن  تحميل الأوزان الي دربناها اً
weights = EfficientNet_V2_S_Weights.DEFAULT
model= efficientnet_v2_s(weights=weights)

# 2.نسوي  Freeze the backbone (feature extractor)
# نقوم نوقف حساب التدرجات لجميع الطبقات عشان نضمن  عدم تدريبها
for param in model.parameters():

    param.requires_grad = False

# 3. نسوي Replace لل classifier head
# EMNIST letters have 26 classes

num_classes = 26

# في EfficientNetV2، رأس التصنيف موجود داخل model.classifier
# نقوم نعرف عدد الميزات الموجوده ة (in_features) من الطبقة الأخيرة الأصلية

in_features =  model.classifier[1].in_features

# نستبدل  طبقة الأخيرة بطبقة جديدة تكون مناسبة  مع 26 كلاس
# الطبقة الجديدة راح يكون لها  ا requires_grad = True
model.classifier[1] = nn.Linear( in_features,  num_classes)

# نودي model الي device (GPU اذا موجود )
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Model loaded and adapted to {num_classes} classes.")

In [ ]:
# Write your code here

import torch

def train_one_epoch (model, train_loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0

    correct = 0
    total = 0

    for images , labels in train_loader:

        labels = labels - 1

        images, labels = images.to(device), labels.to(device)

        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backword and optaimzation

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validate(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            labels = labels - 1

            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            loss = criterion (outputs, labels)

            running_loss += loss.item ()

            _, predicted = torch.max (outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(test_loader)

    accuracy = 100 * correct / total
    return avg_loss, accuracy


In [ ]:
# Write your code here
import matplotlib.pyplot as plt

import torch.optim as optim


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)


num_epochs = 5

train_losses, val_losses = [], []

train_accs, val_accs = [], []

print("Start Train ")
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    #
    train_losses.append(train_loss)

    val_losses.append(val_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss: {train_loss:.4f}, Traing Accurcy: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f}, Valdation Accursy : {val_acc:.2f}%")


plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Traing Losses')
plt.plot(val_losses, label='Valdatin Losses')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()


plt.subplot(1, 2, 2)

plt.plot(train_accs, label='Traing Accursy')
plt.plot(val_accs, label='Valdation Accursy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Write your code here
